In [ ]:
import pathlib
from collections import Counter
import matplotlib.pyplot as plt

import pandas as pd

In [ ]:
PAS_OUT_DIR = pathlib.Path('./filtered_pas')
PAS_OUT_DIR.mkdir(exist_ok=True)

# Read PAS bed marked with DB 

In [ ]:
# Parameters for intersecting with databases
INTERSECT_WINDOW = (50, 25)

INTERSECT_DIR = pathlib.Path(f'Bed_intersect_Up{INTERSECT_WINDOW[0]}bp_Down{INTERSECT_WINDOW[1]}bp')
INTERSECT_DIR

In [ ]:
df_pas_all = pd.read_pickle(INTERSECT_DIR / 'all_peaks_evaluated_intersected.pkl')
print(len(df_pas_all))
df_pas_all.head()

In [ ]:
df_pas_all.pivot_table(index='pas_from', columns='pas_type', aggfunc='size', fill_value=0).head()

# Filter PAS

In [ ]:
# GENCODE: Yes
df_pas_gencode = df_pas_all[df_pas_all['GENCODE']].copy()
df_pas_gencode.pivot_table(index='pas_from', columns='pas_type', aggfunc='size', fill_value=0).head()

In [ ]:
# GENCODE: No, # Matched DB >= 2 Yes
df_pas_db = df_pas_all[(~df_pas_all['GENCODE']) & (df_pas_all['# Anno (without GENCODE)'] >= 2)]
df_pas_db.pivot_table(index='pas_from', columns='pas_type', aggfunc='size', fill_value=0).head()

In [ ]:
# GENCODE: No, # Matched DB == 0
df_pas_zerodb = df_pas_all[(~df_pas_all['GENCODE']) 
                           & (df_pas_all['# Anno (without GENCODE)'] == 0)]
df_pas_zerodb.pivot_table(index='pas_from', columns='pas_type', aggfunc='size', fill_value=0).head()

In [ ]:
# GENCODE: No, # Matched DB == 1
df_pas_onedb = df_pas_all[(~df_pas_all['GENCODE']) 
                           & (df_pas_all['# Anno (without GENCODE)'] == 1)
                           & (~df_pas_all['DeepPASS'])]
df_pas_onedb.pivot_table(index='pas_from', columns='pas_type', aggfunc='size', fill_value=0).head()

In [ ]:
# GENCODE: No, # Matched DB == 1, SCAPTURE DeepPASS: Yes
df_pas_scapture = df_pas_all[(~df_pas_all['GENCODE']) 
                             & (df_pas_all['# Anno (without GENCODE)'] == 1)
                             & df_pas_all['DeepPASS']]
df_pas_scapture.pivot_table(index='pas_from', columns='pas_type', aggfunc='size', fill_value=0).head()

In [ ]:
# Sanity check: these three should have no overlap
for name1, df1 in zip(['gencode', '2db', '0db', '1db', 'scapture'], 
                      [df_pas_gencode, df_pas_db, df_pas_zerodb, df_pas_onedb, df_pas_scapture]):
    for name2, df2 in zip(['gencode', '2db', '0db', '1db', 'scapture'], 
                          [df_pas_gencode, df_pas_db, df_pas_zerodb, df_pas_onedb, df_pas_scapture]):
        if name1 == name2: continue
        print(name1, name2, set(df1.index) & set(df2.index))


In [ ]:
print(Counter(df_pas_gencode['GENCODE']), Counter(df_pas_gencode['DeepPASS']))
print(Counter(df_pas_db['GENCODE']), Counter(df_pas_db['DeepPASS']), Counter(df_pas_db['# Anno (including GENCODE)']))

# Save the final table

In [ ]:
# Final table: 
df_pas_filt = pd.concat([df_pas_gencode, df_pas_db, df_pas_scapture])
print('Percent remain after filtering: ', 100 * len(df_pas_filt) / len(df_pas_all))
df_pas_filt

In [ ]:
df_pas_filt.to_pickle(PAS_OUT_DIR / 'filtered_peaks_evaluated_intersected.pkl.gz')
df_pas_filt.to_csv(PAS_OUT_DIR / 'filtered_peaks_evaluated_intersected.tsv.gz', sep='\t', compression='gzip', index=False)

In [ ]:
df_pas_filt[range(12)]

In [ ]:
for group, df in df_pas_filt.groupby('pas_from'):
    df[range(12)].to_csv(PAS_OUT_DIR / f'{group}.filtered_pas.bed', sep='\t', index=False, header=False)


# Draw number of PAS from each step

In [ ]:
# Draw per region
age_order = [
    "Childhood", 
    "Adolescence", 
    "Young_adult", 
    "Middle_adult", 
    "Late_adult"
]
region_order = '3UTR intron exon CDS 3primeExtended 5UTR'.split()

def sort_by_age_and_celltype(df_pivot, sort_region=True):
    # Split the 'pas_from' into 'age' and 'cell type'
    if sort_region:
        df_pivot = df_pivot[region_order]
    df_pivot = df_pivot.reset_index()
    df_pivot['age'] = df_pivot['pas_from'].str.extract(r'(Childhood|Adolescence|Young_adult|Middle_adult|Late_adult)')
    df_pivot['cell_type'] = df_pivot['pas_from'].str.replace(r'(Childhood|Adolescence|Young_adult|Middle_adult|Late_adult)-', '', regex=True)
    
    # Sort by age and cell type
    df_pivot['age'] = pd.Categorical(df_pivot['age'], categories=age_order, ordered=True)
    df_pivot = df_pivot.sort_values(by=['age', 'cell_type'])
    
    # Restore the sorted index
    df_pivot.set_index('pas_from', inplace=True)
    df_pivot.drop(columns=['age', 'cell_type'], inplace=True)
    
    return df_pivot

def plot_stacked_bar(df_pivot, title, pdf):
    ax = df_pivot.plot(kind='bar', stacked=True, figsize=(8, 2), colormap='tab10', width=0.7)
    plt.title(title)
    plt.xlabel('PAS from')
    plt.ylabel('Counts')
    plt.legend(title='pas_type', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.savefig(pdf, bbox_inches='tight')
    plt.show()


df_pivot_gencode = df_pas_gencode.pivot_table(index='pas_from', columns='pas_type', aggfunc='size', fill_value=0)
df_pivot_db = df_pas_db.pivot_table(index='pas_from', columns='pas_type', aggfunc='size', fill_value=0)
df_pivot_onedb = df_pas_onedb.pivot_table(index='pas_from', columns='pas_type', aggfunc='size', fill_value=0)
df_pivot_scapture = df_pas_scapture.pivot_table(index='pas_from', columns='pas_type', aggfunc='size', fill_value=0)

df_pivot_gencode = sort_by_age_and_celltype(df_pivot_gencode)
df_pivot_db = sort_by_age_and_celltype(df_pivot_db)
df_pivot_onedb = sort_by_age_and_celltype(df_pivot_onedb)
df_pivot_scapture = sort_by_age_and_celltype(df_pivot_scapture)

plot_stacked_bar(df_pivot_gencode, "GENCODE: Yes", 'Num_region_GENCODE.pdf')
plot_stacked_bar(df_pivot_db, "GENCODE: No, Matched DB >= 2", 'Num_region_DB.pdf')
plot_stacked_bar(df_pivot_onedb, "GENCODE: No, Matched DB == 1, DeepPASS == No", 'Num_region_OneDB.pdf')
plot_stacked_bar(df_pivot_scapture, "GENCODE: No, Matched DB == 1, DeepPASS == Yes", 'Num_region_SCAPTUREpdf')

In [ ]:
# Draw per each step

df_counts = pd.DataFrame()
df_counts['GENCODE'] = df_pas_gencode.groupby('pas_from').size()
df_counts['Matched 2 DB'] = df_pas_db.groupby('pas_from').size()
df_counts['Matched 1 DB, filter passed'] = df_pas_scapture.groupby('pas_from').size()

df_counts = sort_by_age_and_celltype(df_counts, sort_region=False)
custom_colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#7f7f7f']
ax = df_counts.plot(kind='bar', stacked=True, figsize=(8, 2), width=0.7, color=custom_colors)
plt.title('Number of PAS from each criterion')
plt.xlabel('PAS from')
plt.ylabel('Counts')
plt.legend(title='pas_type', bbox_to_anchor=(1.01, 1.0), loc='upper left')
plt.savefig('Number_of_PAS_from_each_step.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Draw per each step
df_counts = pd.DataFrame()
df_counts['GENCODE'] = df_pas_gencode.groupby('pas_from').size()
df_counts['Matched 2 DB'] = df_pas_db.groupby('pas_from').size()
df_counts['Matched 1 DB, DeepPASS Yes'] = df_pas_scapture.groupby('pas_from').size()
df_counts['Matched 1 DB, DeepPASS No'] = df_pas_onedb.groupby('pas_from').size()
df_counts['Matched 0 DB'] = df_pas_zerodb.groupby('pas_from').size()

df_counts = sort_by_age_and_celltype(df_counts, sort_region=False)
custom_colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#999999', '#CCCCCC']
ax = df_counts.plot(kind='bar', stacked=True, figsize=(8, 6), width=0.7, color=custom_colors)
plt.title('Number of PAS from each criterion')
plt.xlabel('PAS from')
plt.ylabel('Counts')
plt.legend(title='pas_type', bbox_to_anchor=(1.01, 1.0), loc='upper left')
plt.savefig('Number_of_PAS_from_each_step.pdf', bbox_inches='tight')
plt.show()

# Draw rotated version
from matplotlib.ticker import MultipleLocator
ax = df_counts.plot(kind='barh', stacked=True, figsize=(8, 7), width=0.7, color=custom_colors)
ax.xaxis.set_major_locator(MultipleLocator(20000))  # Adjust based on your data scale
ax.xaxis.set_minor_locator(MultipleLocator(10000))   # Finer minor ticks
plt.title('Number of PAS from each criterion')
plt.ylabel('PAS from')
plt.xlabel('Counts')
plt.legend(title='pas_type', bbox_to_anchor=(1.01, 1.0), loc='upper left')
plt.savefig('Number_of_PAS_from_each_step_barh.pdf', bbox_inches='tight')
plt.show()

In [ ]:
df_counts

# Check current list vs SCAPTURE DeepPASS

In [ ]:
df_pas_all.head(3).T

In [ ]:
df_pas_orgscapture = df_pas_all[(df_pas_all[12]>0) | df_pas_all['DeepPASS']].copy()

df_overlap_counts = pd.DataFrame()
df_overlap_counts['Current list only'] = df_pas_all.loc[df_pas_filt.index.difference(df_pas_orgscapture.index)].groupby('pas_from').size()
df_overlap_counts['Both'] = df_pas_all.loc[df_pas_orgscapture.index.intersection(df_pas_filt.index)].groupby('pas_from').size()
df_overlap_counts['SCAPTURE pipeline only'] = df_pas_all.loc[df_pas_orgscapture.index.difference(df_pas_filt.index)].groupby('pas_from').size()

df_overlap_counts = sort_by_age_and_celltype(df_overlap_counts, sort_region=False)
ax = df_overlap_counts.plot(kind='bar', stacked=True, figsize=(8, 2), width=0.7, color='#1f77b4 #7f4d91 #d62728'.split())
plt.title('Overlap between PAS list from previous pipeline vs current method')
plt.xlabel('PAS from')
plt.ylabel('Counts')
plt.legend(title='pas_type', bbox_to_anchor=(1.05, 1), loc='upper left')
#plt.savefig(pdf, bbox_inches='tight')
plt.show()


In [ ]:
df_overlap_counts.div(df_overlap_counts.sum(axis=1), axis=0)